SETUP AND INPORTS

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from github_workdir.helpers import *            # load_csv_data, create_csv_submission
from src.implementations import *    # your 6 functions from Table 1

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LOAD THE DATA

In [11]:
DATA_PATH = "data/"   # folder containing x_train.csv, y_train.csv, x_test.csv
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print(x_train.shape, x_test.shape, y_train.shape)
print("Fraction of positive labels:", np.mean(y_train == 1))

(328135, 321) (109379, 321) (328135,)
Fraction of positive labels: 0.08830207079403295


MINIMAL PREPOCESSING

In [12]:
# Replace missing values (NaN) by the column median computed on the TRAIN set
median = np.nan_to_num(np.nanmedian(x_train, axis=0))
x_train = np.where(np.isnan(x_train), median, x_train)
x_test = np.where(np.isnan(x_test), median, x_test)

# Standardize with train statistics
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
std[std == 0] = 1                     # avoid division by zero
x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

# Add the offset (bias) column -> tx, as in the labs
tx_train = np.c_[np.ones(x_train.shape[0]), x_train]
tx_test = np.c_[np.ones(x_test.shape[0]), x_test]

# Labels in {0,1} for logistic regression
y = (y_train + 1) / 2

LOCAL TRAIN/VALIDATION SPLIT

In [13]:
def split_data(y, tx, ratio, seed=1):
    """Randomly split the data into train and validation sets."""
    np.random.seed(seed)
    idx = np.random.permutation(len(y))
    cut = int(ratio * len(y))
    return tx[idx[:cut]], y[idx[:cut]], tx[idx[cut:]], y[idx[cut:]]

tx_tr, y_tr, tx_val, y_val = split_data(y, tx_train, ratio=0.8)

TRAIN A MODEL

In [15]:
initial_w = np.zeros(tx_tr.shape[1])
max_iters = 200
gamma = 0.1
lambda_ = 1e-4

w, loss = reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)
print("train loss:", loss)

train loss: 0.2426247628037984


EVALUATE ON VALIDATION SET

In [16]:
def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-t))

def predict_labels(w, tx, threshold=0.5):
    """Return predictions in {-1, 1} (format required by the submission)."""
    return np.where(sigmoid(tx @ w) >= threshold, 1, -1)

def f1_score(y_true, y_pred):
    """F1 score for labels in {-1, 1}."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return 2 * tp / (2 * tp + fp + fn + 1e-12)

y_val_pm = 2 * y_val - 1             # back to {-1, 1}
y_pred_val = predict_labels(w, tx_val)
print("accuracy:", np.mean(y_pred_val == y_val_pm))
print("F1 score:", f1_score(y_val_pm, y_pred_val))

accuracy: 0.9157511390129063
F1 score: 0.13161614575153133


CREATE THE SUBMISSION

In [18]:
y_pred = predict_labels(w, tx_test)
create_csv_submission(test_ids, y_pred, "submission1.csv")